In [19]:
import optuna
import os
import pickle

from dotenv import load_dotenv
from sklearn.base import clone
from sqlalchemy import create_engine

from customer_analytics.churn_prediction.database import ClassifierDataIterator
from customer_analytics.churn_prediction.modelling import suggest_parameters, fit_scaler, train_model, evaluate_model

In [20]:
load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))
seed = 123
n_trials = 20
train_sample_size = 200_000
validation_sample_size = 50_000
batch_size = 20_000

In [21]:
with open("../data/classifier_techniques.pkl", "rb") as f:
    classifier_techniques = pickle.load(f)

In [22]:
classifier_techniques["logistic_regression"]["params_distribution"] = {
    "alpha": {
        "type": "float",
        "low": 1e-6,
        "high": 1e-2,
        "log": True
    },
    "l1_ratio": {
        "type": "float",
        "low": 0.0,
        "high": 1.0
    },
    "learning_rate": {
        "type": "categorical",
        "choices": ["optimal", "adaptive", "invscaling"]
    },
    "eta0": {
        "type": "float",
        "low": 1e-4,
        "high": 1e-1,
        "log": True
    }
}

classifier_techniques["support_vector_machine"]["params_distribution"] = {
    "alpha": {
        "type": "float",
        "low": 1e-6,
        "high": 1e-2,
        "log": True
    },
    "l1_ratio": {
        "type": "float",
        "low": 0.0,
        "high": 1.0
    },
    "learning_rate": {
        "type": "categorical",
        "choices": ["optimal", "adaptive", "invscaling"]
    },
    "eta0": {
        "type": "float",
        "low": 1e-4,
        "high": 1e-1,
        "log": True
    }
}

classifier_techniques["mlp_classifier"]["params_distribution"] = {
    "hidden_layer_sizes": {
        "type": "categorical",
        "choices": [
            (32,),
            (64,),
            (128,),
            (32, 32),
            (64, 64),
            (128, 64)
        ]
    },
    "activation": {
        "type": "categorical",
        "choices": ["relu", "tanh"]
    },
    "alpha": {
        "type": "float",
        "low": 1e-6,
        "high": 1e-2,
        "log": True
    },
    "learning_rate_init": {
        "type": "float",
        "low": 1e-4,
        "high": 1e-2,
        "log": True
    }
}

classifier_techniques["naive_bayes"]["params_distribution"] = {
    "var_smoothing": {
        "type": "float",
        "low": 1e-12,
        "high": 1e-7,
        "log": True
    }
}

classifier_techniques["xgboost"]["params_distribution"] = {
    "n_estimators": {
        "type": "categorical",
        "choices": [100, 200, 300, 500, 750, 1000]
    },
    "max_depth": {
        "type": "categorical",
        "choices": [3, 4, 5, 6, 8]
    },
    "learning_rate": {
        "type": "float",
        "low": 0.01,
        "high": 0.3,
        "log": True
    },
    "subsample": {
        "type": "float",
        "low": 0.6,
        "high": 1.0
    },
    "colsample_bytree": {
        "type": "float",
        "low": 0.6,
        "high": 1.0
    },
    "min_child_weight": {
        "type": "float",
        "low": 1,
        "high": 20,
        "log": True
    },
    "gamma": {
        "type": "float",
        "low": 0,
        "high": 5
    },
    "reg_alpha": {
        "type": "float",
        "low": 1e-5,
        "high": 10,
        "log": True
    },
    "reg_lambda": {
        "type": "float",
        "low": 1e-3,
        "high": 100,
        "log": True
    }
}

In [23]:
def objective(
    trial,
    classifier_techniques,
    classifier_name,
    engine
):

    technique = classifier_techniques[classifier_name]

    params = suggest_parameters(
        trial,
        classifier_techniques,
        classifier_name
    )

    model = clone(technique["method"])
    model.set_params(**params)

    train_iterator = ClassifierDataIterator(
        engine=engine,
        split="train",
        batch_size=batch_size,
        sample_size=train_sample_size
    )

    validation_iterator = ClassifierDataIterator(
        engine=engine,
        split="validation",
        batch_size=batch_size,
        sample_size=validation_sample_size
    )

    scaler = None

    if technique["requires_scaling"]:
        scaler = fit_scaler(train_iterator)

    epoch = 0
    max_epochs = 10

    while epoch < max_epochs:

        model = train_model(
            model=model,
            train_iterator=train_iterator,
            scaler=scaler,
            epoch=epoch
        )

        score = evaluate_model(
            model=model,
            validation_iterator=validation_iterator,
            scaler=scaler
        )

        trial.report(score, step=epoch)

        if trial.should_prune():
            raise optuna.TrialPruned()
        
        epoch += 1

    return score

In [24]:
studies = {}

sampler = optuna.samplers.TPESampler(seed=seed)

pruner = optuna.pruners.HyperbandPruner(
    min_resource=1,
    max_resource=10,
    reduction_factor=3
)

for classifier_name in classifier_techniques:

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        pruner=pruner,
        study_name=f"churn_classifier_{classifier_name}"
    )

    study.optimize(
        lambda trial, name=classifier_name: objective(
            trial,
            classifier_techniques,
            name,
            engine
        ),
        n_jobs=1,
        n_trials=n_trials,
        show_progress_bar=True
    )

    studies[classifier_name] = study

[I 2026-11-20 12:37:37,057] A new study created in memory with name: churn_classifier_logistic_regression


  0%|          | 0/20 [00:00<?, ?it/s]

Fitting scaler:   0%|          | 0/10 [00:00<?, ?it/s]

Training epoch 1:   0%|          | 0/10 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

Training epoch 2:   0%|          | 0/10 [00:00<?, ?it/s]

... (2 outputs left out)

Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

Training epoch 4:   0%|          | 0/10 [00:00<?, ?it/s]

[W 2026-11-20 12:41:21,622] Trial 0 failed with parameters: {'alpha': 0.0006107686573608485, 'l1_ratio': 0.28613933495037946, 'learning_rate': 'invscaling', 'eta0': 0.0018591711878786364} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\Charlie\Documents\coding\python\projects\customer-analytics\.venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\Charlie\AppData\Local\Temp\ipykernel_11836\656575884.py", line 21, in <lambda>
    lambda trial, name=classifier_name: objective(
                                        ^^^^^^^^^^
  File "C:\Users\Charlie\AppData\Local\Temp\ipykernel_11836\1299669071.py", line 43, in objective
    model = train_model(
            ^^^^^^^^^^^^
  File "C:\Users\Charlie\Documents\coding\python\projects\customer-analytics\src\customer_analytics\churn_prediction\modelling.py", line 63, in train_model
 

KeyboardInterrupt: 

In [ ]:
with open("../data/studies.pkl", "wb") as file:
    pickle.dump(studies, file)